In [1]:
import xarray as xr
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import os
from glob import glob
from datetime import datetime, timedelta

In [2]:
# Configuration
SAVE_DIR = "WWTP_analysis_longrun_with_bottom"
os.makedirs(SAVE_DIR, exist_ok=True)

# Define base paths for different scenarios
BASE_PATHS = {
    "Baseline": "/ocean/atall/MOAD/Model/202410b/oxygen_forZ2018/",
    "WWTP_noNut": "/ocean/atall/MOAD/Model/202410b/wastewatero2_freshwater", 
    "WWTP_withNut": "/ocean/atall/MOAD/Model/202410b/wastewatero2"
}

In [3]:
# Load mesh mask for bottom analysis
print("Loading mesh mask...")
MESH_MASK_PATH = '/ocean/atall/MOAD/grid/mesh_mask_202310b.nc'
meshmask = xr.open_dataset(MESH_MASK_PATH)
tmask = meshmask.tmask
mbathy = meshmask.mbathy

print(f"Mesh mask loaded: tmask shape {tmask.shape}, mbathy shape {mbathy.shape}")


Loading mesh mask...
Mesh mask loaded: tmask shape (1, 40, 898, 398), mbathy shape (1, 898, 398)


In [4]:
# Generate date range for 7 months (January to July 2018)
def generate_date_range():
    """Generate all dates from 01jan18 to 31jul18"""
    start_date = datetime(2018, 7, 25)
    end_date = datetime(2018, 7, 31)
    dates = []
    
    current_date = start_date
    while current_date <= end_date:
        date_str = current_date.strftime("%d%b%y").lower()
        dates.append(date_str)
        current_date += timedelta(days=1)
    
    return dates

# Get all dates
DATES = generate_date_range()
print(f"Processing {len(DATES)} days from {DATES[0]} to {DATES[-1]}")

def find_netcdf_files(base_path, date, file_type):
    """Find NetCDF files for a specific date and type"""
    patterns = [
        f"{base_path}/{date}/SalishSea_1d_*_{file_type}_T.nc"
    ]
    
    for pattern in patterns:
        files = glob(pattern)
        if files:
            return files
    return []

def load_scenario_data(scenario_name, file_type):
    """Load data for a specific scenario and file type across all dates"""
    base_path = BASE_PATHS[scenario_name]
    all_files = []
    
    print(f"Loading {scenario_name} {file_type} data...")
    
    for date in tqdm(DATES, desc=f"{scenario_name}_{file_type}"):
        files = find_netcdf_files(base_path, date, file_type)
        if files:
            all_files.extend(files)
        else:
            print(f"  Warning: No {file_type} files found for {date}")
    
    if not all_files:
        print(f"  No {file_type} files found for {scenario_name}")
        return None
    
    try:
        print(f"  Opening {len(all_files)} files...")
        ds = xr.open_mfdataset(all_files, combine='by_coords', parallel=True)
        print(f"  Successfully loaded {scenario_name} {file_type} data")
        return ds
    except Exception as e:
        print(f"  Error loading {scenario_name} {file_type}: {e}")
        try:
            ds = xr.open_mfdataset(all_files, combine='nested', concat_dim='time_counter', parallel=True)
            print(f"  Successfully loaded with alternative method")
            return ds
        except Exception as e2:
            print(f"  Alternative method also failed: {e2}")
            return None

def load_all_data():
    """Load all data for all scenarios and types"""
    all_data = {}
    
    for scenario in BASE_PATHS.keys():
        print(f"\n=== Loading {scenario} scenario ===")
        scenario_data = {}
        
        # Load chemical data
        chem_ds = load_scenario_data(scenario, "chem")
        if chem_ds is not None:
            scenario_data["chem"] = chem_ds
        
        # Load biological data  
        biol_ds = load_scenario_data(scenario, "biol")
        if biol_ds is not None:
            scenario_data["biol"] = biol_ds
            
        # Load diagnostic data
        diag_ds = load_scenario_data(scenario, "diag") 
        if diag_ds is not None:
            scenario_data["diag"] = diag_ds
            
        all_data[scenario] = scenario_data
    
    return all_data


Processing 7 days from 25jul18 to 31jul18


In [5]:
# Load source locations
def load_source_locations(wwtp_file_path):
    """Load source locations from flux file"""
    print(f"Loading source locations from {wwtp_file_path}")
    ds_sources = xr.open_dataset(wwtp_file_path)
    
    # Identify active sources (non-zero flux)
    flux_nonzero = ds_sources["flux"].max(dim="time_counter") > 0
    source_indices = np.argwhere(flux_nonzero.values)
    
    # Get coordinates
    lon = ds_sources["nav_lon"].values
    lat = ds_sources["nav_lat"].values
    
    print(f"Found {len(source_indices)} active sources")
    
    return ds_sources, source_indices, lon, lat


In [6]:
# Enhanced data extraction functions with bottom analysis
def get_bottom_depth(y, x):
    """Get bottom depth level using mbathy"""
    try:
        # mbathy gives the number of wet levels at each point
        # Use .item() to convert 0-d array to scalar
        bottom_level = mbathy.isel(y=y, x=x).item() - 1  # Convert to 0-based indexing
        return max(0, int(bottom_level))  # Ensure non-negative integer
    except Exception as e:
        print(f"Error getting bottom depth at ({y},{x}): {e}")
        return 0  # Default to surface if error


In [7]:

def extract_local_mean(ds, var_name, y, x, dx=2, depth_level=0):
    """Extract local mean around point (y,x) at specified depth level"""
    try:
        if var_name not in ds.variables:
            print(f"Variable {var_name} not found in dataset")
            return None
            
        var_data = ds[var_name]
        
        # Handle different dimension structures
        if 'deptht' in var_data.dims and 'y' in var_data.dims and 'x' in var_data.dims:
            # 4D data: time, deptht, y, x
            y_min = max(0, y-dx)
            y_max = min(var_data.sizes['y'], y+dx+1)
            x_min = max(0, x-dx)
            x_max = min(var_data.sizes['x'], x+dx+1)
            
            sub = var_data.isel(y=slice(y_min, y_max), x=slice(x_min, x_max))
            
            # Extract at specified depth level and average over space
            if 'deptht' in sub.dims:
                depth_data = sub.isel(deptht=depth_level)
                local_mean = depth_data.mean(dim=['y', 'x'])
            else:
                local_mean = sub.mean(dim=['y', 'x'])
                
        elif 'y' in var_data.dims and 'x' in var_data.dims:
            # 3D data: time, y, x  
            y_min = max(0, y-dx)
            y_max = min(var_data.sizes['y'], y+dx+1)
            x_min = max(0, x-dx)
            x_max = min(var_data.sizes['x'], x+dx+1)
            
            sub = var_data.isel(y=slice(y_min, y_max), x=slice(x_min, x_max))
            local_mean = sub.mean(dim=['y', 'x'])
        else:
            print(f"Unexpected dimensions for {var_name}: {list(var_data.dims)}")
            return None
            
        return local_mean.values
        
    except Exception as e:
        print(f"Error extracting {var_name} at ({y},{x}), depth {depth_level}: {e}")
        return None

In [8]:
def extract_surface_and_bottom(ds, var_name, y, x, dx=2):
    """Extract both surface and bottom values"""
    surface_data = extract_local_mean(ds, var_name, y, x, dx, depth_level=0)
    
    # Get bottom depth
    bottom_level = get_bottom_depth(y, x)
    bottom_data = extract_local_mean(ds, var_name, y, x, dx, depth_level=bottom_level)
    
    return {
        'surface': surface_data,
        'bottom': bottom_data,
        'bottom_level': bottom_level
    }

def extract_diag_terms(ds, terms_dict, y, x, dx=2, depth_level=0):
    """Extract diagnostic terms local mean around point (y,x) at specified depth level"""
    diag_data = {}
    
    for term_name, var_name in terms_dict.items():
        data = extract_local_mean(ds, var_name, y, x, dx, depth_level)
        if data is not None:
            diag_data[term_name] = data
    
    return diag_data

In [9]:
# Main processing function with bottom analysis
def process_all_sources(all_data, source_indices, lon, lat):
    """Process all sources and extract surface + bottom data including diagnostics"""
    results = {}
    bottom_depths = {}  # Store bottom depths for each source
    
    # Define variables to extract
    variables = {
        "O2": {
            "var_name": "dissolved_oxygen",
            "file_type": "chem"
        },
        "NO3": {
            "var_name": "nitrate", 
            "file_type": "biol"
        },
        "NH3": {
            "var_name": "ammonium",
            "file_type": "biol"
        },
        "Z2": {
            "var_name": "mesozooplankton",
            "file_type": "biol"
        }
    }
    
    diag_terms = {
        "PRD_O2": "PRD_O2",
        "REM_O2": "REM_O2", 
        "BIO_O2": "BIO_O2",
        "PHS_O2": "PHS_O2",
        "dDO_dt": "RATE_O2"
    }
    
    print(f"\nProcessing {len(source_indices)} sources (surface + bottom + diagnostics)...")
    
    for idx, (y, x) in tqdm(enumerate(source_indices), total=len(source_indices)):
        res_source = {}
        
        # Store bottom depth for this source
        bottom_depth_level = get_bottom_depth(y, x)
        bottom_depths[idx] = bottom_depth_level
        
        # Extract main variables for each scenario (surface + bottom)
        for var_name, var_info in variables.items():
            var_surface_series = []
            var_bottom_series = []
            
            for scenario in BASE_PATHS.keys():
                if var_info["file_type"] in all_data[scenario]:
                    ds = all_data[scenario][var_info["file_type"]]
                    depth_data = extract_surface_and_bottom(ds, var_info["var_name"], y, x)
                    
                    if depth_data['surface'] is not None:
                        var_surface_series.append(depth_data['surface'])
                    else:
                        var_surface_series.append(None)
                    
                    if depth_data['bottom'] is not None:
                        var_bottom_series.append(depth_data['bottom'])
                    else:
                        var_bottom_series.append(None)
                else:
                    var_surface_series.append(None)
                    var_bottom_series.append(None)
            
            # Handle cases where some scenarios have no data
            valid_surface = [s for s in var_surface_series if s is not None]
            valid_bottom = [s for s in var_bottom_series if s is not None]
            
            if valid_surface:
                for i, series in enumerate(var_surface_series):
                    if series is None and valid_surface:
                        var_surface_series[i] = np.full_like(valid_surface[0], np.nan)
            
            if valid_bottom:
                for i, series in enumerate(var_bottom_series):
                    if series is None and valid_bottom:
                        var_bottom_series[i] = np.full_like(valid_bottom[0], np.nan)
            
            res_source[var_name] = {
                'surface': var_surface_series,  # [Baseline, noNut, withNut]
                'bottom': var_bottom_series     # [Baseline, noNut, withNut]
            }
        
        # Extract diagnostic terms (surface AND bottom)
        diag_surface_series = []
        diag_bottom_series = []
        
        for scenario in BASE_PATHS.keys():
            if "diag" in all_data[scenario]:
                ds = all_data[scenario]["diag"]
                
                # Extract surface diagnostics
                diag_surface = extract_diag_terms(ds, diag_terms, y, x, dx=2, depth_level=0)
                diag_surface_series.append(diag_surface)
                
                # Extract bottom diagnostics
                diag_bottom = extract_diag_terms(ds, diag_terms, y, x, dx=2, depth_level=bottom_depth_level)
                diag_bottom_series.append(diag_bottom)
            else:
                diag_surface_series.append({})
                diag_bottom_series.append({})
        
        res_source["diag"] = {
            'surface': diag_surface_series,
            'bottom': diag_bottom_series
        }
        results[idx] = res_source
    
    return results, variables, diag_terms, bottom_depths



In [10]:
# Create DataFrames with depth information
def create_dataframes(results, source_indices, lon, lat, variables, diag_terms, bottom_depths):
    """Create main and diagnostic DataFrames with depth information"""
    records_main = []
    records_diag = []
    
    # Get source coordinates
    yy, xx = zip(*source_indices)
    
    # Handle different lon/lat dimension structures
    if len(lon.shape) == 2:
        lon_sources = lon[yy, xx]
        lat_sources = lat[yy, xx]
    else:
        lon_sources = lon[0, yy, xx]
        lat_sources = lat[0, yy, xx]
    
    print("Creating DataFrames with depth information (including bottom diagnostics)...")
    
    for idx, res in tqdm(results.items(), total=len(results)):
        lon_i = lon_sources[idx] if hasattr(lon_sources, '__len__') else lon_sources
        lat_i = lat_sources[idx] if hasattr(lat_sources, '__len__') else lat_sources
        bottom_depth = bottom_depths[idx]
        
        # Process main variables (surface and bottom)
        for var_name in variables.keys():
            if var_name in res:
                # Process surface data
                for sim_idx, sim_label in enumerate(["Baseline", "WWTP_noNut", "WWTP_withNut"]):
                    if (sim_idx < len(res[var_name]['surface']) and 
                        res[var_name]['surface'][sim_idx] is not None):
                        series = res[var_name]['surface'][sim_idx]
                        for t, val in enumerate(series):
                            records_main.append({
                                "source_id": idx,
                                "lon": float(lon_i),
                                "lat": float(lat_i),
                                "time_index": t,
                                "variable": var_name,
                                "simulation": sim_label,
                                "depth_type": "surface",
                                "depth_level": 0,
                                "bottom_depth": int(bottom_depth),
                                "value": float(val) if not np.isnan(val) else np.nan
                            })
                
                # Process bottom data
                for sim_idx, sim_label in enumerate(["Baseline", "WWTP_noNut", "WWTP_withNut"]):
                    if (sim_idx < len(res[var_name]['bottom']) and 
                        res[var_name]['bottom'][sim_idx] is not None):
                        series = res[var_name]['bottom'][sim_idx]
                        for t, val in enumerate(series):
                            records_main.append({
                                "source_id": idx,
                                "lon": float(lon_i),
                                "lat": float(lat_i),
                                "time_index": t,
                                "variable": var_name,
                                "simulation": sim_label,
                                "depth_type": "bottom",
                                "depth_level": int(bottom_depth),
                                "bottom_depth": int(bottom_depth),
                                "value": float(val) if not np.isnan(val) else np.nan
                            })
        
        # Process diagnostic terms (surface AND bottom)
        if "diag" in res:
            # Surface diagnostics
            for sim_idx, sim_label in enumerate(["Baseline", "WWTP_noNut", "WWTP_withNut"]):
                if sim_idx < len(res["diag"]['surface']):
                    diag_data = res["diag"]['surface'][sim_idx]
                    for diag_name, diag_series in diag_data.items():
                        for t, val in enumerate(diag_series):
                            records_diag.append({
                                "source_id": idx,
                                "lon": float(lon_i),
                                "lat": float(lat_i),
                                "time_index": t,
                                "diag_term": diag_name,
                                "simulation": sim_label,
                                "depth_type": "surface",
                                "depth_level": 0,
                                "bottom_depth": int(bottom_depth),
                                "value": float(val) if not np.isnan(val) else np.nan
                            })
            
            # Bottom diagnostics
            for sim_idx, sim_label in enumerate(["Baseline", "WWTP_noNut", "WWTP_withNut"]):
                if sim_idx < len(res["diag"]['bottom']):
                    diag_data = res["diag"]['bottom'][sim_idx]
                    for diag_name, diag_series in diag_data.items():
                        for t, val in enumerate(diag_series):
                            records_diag.append({
                                "source_id": idx,
                                "lon": float(lon_i),
                                "lat": float(lat_i),
                                "time_index": t,
                                "diag_term": diag_name,
                                "simulation": sim_label,
                                "depth_type": "bottom",
                                "depth_level": int(bottom_depth),
                                "bottom_depth": int(bottom_depth),
                                "value": float(val) if not np.isnan(val) else np.nan
                            })
    
    # Create DataFrames
    df_main = pd.DataFrame(records_main)
    df_diag = pd.DataFrame(records_diag) if records_diag else pd.DataFrame()
    
    print(f"Main DataFrame: {df_main.shape}")
    print(f"Diagnostics DataFrame: {df_diag.shape}")
    
    # Print depth statistics
    if not df_main.empty and 'bottom_depth' in df_main.columns:
        surface_count = len(df_main[df_main['depth_type'] == 'surface'])
        bottom_count = len(df_main[df_main['depth_type'] == 'bottom'])
        
        # Ensure bottom_depth is numeric and calculate mean properly
        df_main['bottom_depth'] = pd.to_numeric(df_main['bottom_depth'], errors='coerce')
        avg_bottom_depth = df_main['bottom_depth'].mean()
        
        print(f"Surface records: {surface_count}, Bottom records: {bottom_count}")
        
        # Check if avg_bottom_depth is a scalar before formatting
        if hasattr(avg_bottom_depth, '__len__') and len(avg_bottom_depth) > 1:
            print(f"Average bottom depth level: {avg_bottom_depth[0]:.1f}")
        else:
            print(f"Average bottom depth level: {float(avg_bottom_depth):.1f}")
    
    if not df_diag.empty:
        diag_surface_count = len(df_diag[df_diag['depth_type'] == 'surface'])
        diag_bottom_count = len(df_diag[df_diag['depth_type'] == 'bottom'])
        print(f"Diagnostics - Surface: {diag_surface_count}, Bottom: {diag_bottom_count}")
    
    return df_main, df_diag

In [11]:
# Main execution
def main():
    """Main execution function"""
    print("=== 6-Month WWTP Analysis with Bottom Data ===")
    
    # Update these paths to your actual data locations
    FLUX_FILE_PATH = "/ocean/atall/MOAD/Obs/PugetSound/WWTP/wastewater_20251118.nc"
    
    try:
        # 1. Load all data
        print("\n1. Loading data...")
        all_data = load_all_data()
        
        # Check if we have all required data
        missing_scenarios = []
        for scenario in BASE_PATHS.keys():
            if scenario not in all_data or not all_data[scenario]:
                missing_scenarios.append(scenario)
        
        if missing_scenarios:
            print(f"Warning: Missing data for scenarios: {missing_scenarios}")
            if len(missing_scenarios) == len(BASE_PATHS):
                print("No data loaded. Exiting.")
                return None, None, None
        
        # 2. Load source locations
        print("\n2. Loading source locations...")
        ds_sources, source_indices, lon, lat = load_source_locations(FLUX_FILE_PATH)
        
        if len(source_indices) == 0:
            print("No active sources found. Exiting.")
            return None, None, None
        
        # 3. Process all sources (surface + bottom)
        print("\n3. Processing sources (surface + bottom)...")
        results, variables, diag_terms, bottom_depths = process_all_sources(all_data, source_indices, lon, lat)
        
        if not results:
            print("No results generated. Exiting.")
            return None, None, None
        
        # 4. Create DataFrames
        print("\n4. Creating DataFrames...")
        df_main, df_diag = create_dataframes(results, source_indices, lon, lat, variables, diag_terms, bottom_depths)
        
        if df_main.empty:
            print("No data in main DataFrame. Exiting.")
            return None, None, None
        
        # 5. Save to CSV
        print("\n5. Saving results...")
        df_main.to_csv(os.path.join(SAVE_DIR, "WWTP_local_series_with_depth.csv"), index=False)
        if not df_diag.empty:
            df_diag.to_csv(os.path.join(SAVE_DIR, "WWTP_diagnostics.csv"), index=False)
        
        # Save bottom depths
        bottom_depths_df = pd.DataFrame.from_dict(bottom_depths, orient='index', columns=['bottom_depth_level'])
        bottom_depths_df.to_csv(os.path.join(SAVE_DIR, "source_bottom_depths.csv"))
        
        print(f"Saved files to {SAVE_DIR}:")
        print(f"  - WWTP_local_series_with_depth.csv")
        print(f"  - source_bottom_depths.csv")
        if not df_diag.empty:
            print(f"  - WWTP_diagnostics.csv")
        
        return df_main, df_diag, bottom_depths
        
    except Exception as e:
        print(f"Error in main execution: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None

In [12]:
# Run the main function
if __name__ == "__main__":
    df_main, df_diag, bottom_depths = main()
    
    if df_main is not None:
        print(f"\n=== Analysis Complete ===")
        print(f"Processed data for {len(df_main['source_id'].unique())} sources")
        print(f"Time steps: {len(df_main['time_index'].unique())}")
        print(f"Variables: {list(df_main['variable'].unique())}")
        print(f"Depth analysis: Surface + Bottom")
        
        # You can continue with your analysis using df_main and df_diag
        print("\Can now run analysis functions on the DataFrames:")
        print("  - df_main: Main data with surface and bottom values")
        print("  - df_diag: Diagnostic data")
    else:
        print("\n=== Analysis Failed ===")
        print("Check the error messages above and verify your data paths.")

=== 6-Month WWTP Analysis with Bottom Data ===

1. Loading data...

=== Loading Baseline scenario ===
Loading Baseline chem data...


Baseline_chem: 100%|██████████| 7/7 [00:00<00:00, 444.40it/s]

  Opening 7 files...


  Successfully loaded Baseline chem data
Loading Baseline biol data...


Baseline_biol: 100%|██████████| 7/7 [00:00<00:00, 1617.91it/s]

...

  Successfully loaded WWTP_withNut diag data

2. Loading source locations...
Loading source locations from /ocean/atall/MOAD/Obs/PugetSound/WWTP/wastewater_20251118.nc
Found 98 active sources

3. Processing sources (surface + bottom)...

Processing 98 sources (surface + bottom + diagnostics)...


100%|██████████| 98/98 [02:16<00:00,  1.39s/it] 



4. Creating DataFrames...
Creating DataFrames with depth information (including bottom diagnostics)...


100%|██████████| 98/98 [00:00<00:00, 1086.46it/s]


Main DataFrame: (16464, 10)
Diagnostics DataFrame: (20580, 10)
Surface records: 8232, Bottom records: 8232
Average bottom depth level: 15.6
Diagnostics - Surface: 10290, Bottom: 10290

5. Saving results...
Saved files to WWTP_analysis_longrun_with_bottom:
  - WWTP_local_series_with_depth.csv
  - source_bottom_depths.csv
  - WWTP_diagnostics.csv

=== Analysis Complete ===
Processed data for 98 sources
Time steps: 7
Variables: ['O2', 'NO3', 'NH3', 'Z2']
Depth analysis: Surface + Bottom
\Can now run analysis functions on the DataFrames:
  - df_main: Main data with surface and bottom values
  - df_diag: Diagnostic data
